# Lab 2A — An Agent That Needs Two Sources to Answer

In [Lab 1B](#) the agent had two toy tools and a dictionary of fake orders.
Here it gets **real sources**: policy documents and live order data, both from
Unity Catalog.

The interesting part is not that it works. It is **how the code is arranged**.

| Cell group | What you do |
|---|---|
| 0–1 | Install, and load the policy documents from Unity Catalog. |
| 2–3 | Write `core.py` — the agent logic — and **prove it mentions Databricks nowhere.** |
| 4–7 | Write four adapters that do mention Databricks. |
| 8 | Wire them together and ask a question that needs **both** sources. |
| 9 | Read which sources it actually touched. |

**Compute:** Serverless.

## 0. Install

Serverless has the Databricks SDK and Spark, but not `openai`.

In [0]:
%pip install -q openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Load the policy documents

These are real rows in Unity Catalog. Note the **`audience`** column — some of
these documents are `agent_only` and must never reach a customer.

In [0]:
docs = [r.asDict() for r in spark.sql("""
    SELECT doc_id, title, category, audience, body
    FROM agents_labs.retail.support_docs
    ORDER BY doc_id
""").collect()]

print(f"{len(docs)} documents\n")
for d in docs:
    print(f"  {d['doc_id']}  {d['audience']:11} {d['title']}")

7 documents

  DOC-001  customer    Returns and refunds policy
  DOC-002  agent_only  Lost in transit procedure
  DOC-003  customer    Delivery timescales by region
  DOC-004  customer    Warranty coverage
  DOC-005  customer    Billing, invoices and payment terms
  DOC-006  agent_only  Refund authority limits
  DOC-007  customer    Bulk and fit-out orders


## 2. The contract: three Protocols

Before any implementation, decide what the agent is *allowed to assume*.

| Protocol | The promise |
|---|---|
| `Retriever` | *"I can find relevant documents."* Not how. |
| `ToolBox` | *"I can describe my tools and run one."* Not what they touch. |
| `Chat` | *"I can complete a conversation, optionally with tools."* Not which model. |

Anything satisfying these three can drive the agent. That is what makes
[Lab 2B](#) a one-line change.

## 3. Write the core — to a file, on purpose

`%%writefile` writes this cell to `core.py` instead of running it.

We use a **file** rather than a cell so the next cell can do something a cell
cannot: check what this code is allowed to import.

In [0]:
%%writefile core.py
"""The portable core. Knows nothing about Databricks, or any model."""
from __future__ import annotations
import json
from typing import Protocol, Any


class Retriever(Protocol):
    def search(self, query: str, k: int = 3, **filters: Any) -> list[dict]: ...
    @property
    def name(self) -> str: ...


class ToolBox(Protocol):
    def schemas(self) -> list[dict]: ...
    def call(self, name: str, args: dict) -> dict: ...


class Chat(Protocol):
    def complete(self, messages: list[dict], tools: list[dict] | None) -> Any: ...
    @property
    def model(self) -> str: ...


SYSTEM = """You are a customer support agent for an office furniture retailer.

You have two sources and you usually need both:
  * search_policy  - the company's written policies (delivery, returns, warranty)
  * the order tools - live facts about a specific order and its customer

Never state a policy you have not retrieved, and never state an order fact you
have not looked up. If answering needs both, use both before you answer.
Cite the policy document id you relied on, like [DOC-003]."""

MAX_STEPS = 6


def text_of(message) -> str:
    """Claude endpoints return content as a list of typed blocks, not a string."""
    c = getattr(message, "content", None)
    if c is None:
        return ""
    if isinstance(c, str):
        return c
    out = []
    for b in c:
        b = b if isinstance(b, dict) else getattr(b, "__dict__", {})
        if b.get("type") == "text" and b.get("text"):
            out.append(b["text"])
    return "\n".join(out).strip()


def run(question: str, *, chat: Chat, retriever: Retriever, tools: ToolBox,
        verbose: bool = True) -> dict:
    """One turn of the agent. Returns the answer plus which sources it touched."""
    schemas = [{
        "type": "function",
        "function": {
            "name": "search_policy",
            "description": "Search the company policy documents. Use for anything "
                           "about delivery times, returns, refunds, warranty or billing.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "audience": {"type": "string",
                                 "description": "customer or agent_only. Omit for customer."},
                },
                "required": ["query"],
            },
        },
    }] + tools.schemas()

    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    used = {"retrieval": [], "tools": []}

    for step in range(1, MAX_STEPS + 1):
        msg = chat.complete(messages, schemas)
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"  step {step}: answer")
            return {"answer": text_of(msg), "used": used,
                    "retriever": retriever.name, "model": chat.model}

        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            if name == "search_policy":
                hits = retriever.search(args["query"], k=3,
                                        audience=args.get("audience", "customer"))
                used["retrieval"] += [h["doc_id"] for h in hits]
                result = {"results": [{"doc_id": h["doc_id"], "title": h["title"],
                                       "text": h["text"], "score": round(h["score"], 4)}
                                      for h in hits]}
                if verbose:
                    print(f"  step {step}: search_policy({args['query']!r}) "
                          f"-> {[h['doc_id'] for h in hits]}")
            else:
                result = tools.call(name, args)
                used["tools"].append(name)
                if verbose:
                    print(f"  step {step}: {name}({json.dumps(args)}) -> "
                          f"{json.dumps(result, default=str)[:110]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, default=str)})

    return {"answer": "(no answer within MAX_STEPS)", "used": used,
            "retriever": retriever.name, "model": chat.model}

Overwriting core.py


### Now prove the claim

The cell above claims to be portable. **Check it**, rather than believing it.

Two checks, and the gap between them is the lesson:

| Check | What it proves |
|---|---|
| **AST imports** | what the module actually depends on — the real test |
| **grep** | what the text contains — blunt, and it produces a false positive here |

In [0]:
import ast, hashlib, re, sys

src = open("core.py").read()

# ---- the real proof: what does this module actually import? ----------------
tree = ast.parse(src)
imports = set()
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        imports |= {a.name.split(".")[0] for a in node.names}
    elif isinstance(node, ast.ImportFrom) and node.module:
        imports.add(node.module.split(".")[0])

external = sorted(i for i in imports if i not in sys.stdlib_module_names)

print("  imports    :", sorted(imports))
print("  NOT stdlib :", external or "none")
print("  verdict    :", "PORTABLE" if not external else "NOT PORTABLE")

# ---- the blunt check, and why it is blunt ---------------------------------
print("\n  grep for platform words:")
for w in ["databricks", "openai", "spark", "dbutils", "mlflow"]:
    nums = [n for n, l in enumerate(src.splitlines(), 1) if re.search(w, l, re.I)]
    print(f"    {w:12} {len(nums)} hit(s){'  line ' + str(nums) if nums else ''}")

hits = [l.strip() for l in src.splitlines() if re.search("databricks", l, re.I)]
if hits:
    print("\n  That hit, in full:")
    for l in hits:
        print(f"    {l}")
    print("\n  It is the docstring SAYING it knows nothing about Databricks.")
    print("  grep cannot tell a comment from a dependency. The AST check can.")

print(f"\n  sha256[:16] = {hashlib.sha256(src.encode()).hexdigest()[:16]}")
print("  Lab 2B recreates this exact file and checks the hash matches.")

  imports    : ['__future__', 'json', 'typing']
  NOT stdlib : none
  verdict    : PORTABLE

  grep for platform words:
    databricks   1 hit(s)  line [1]
    openai       0 hit(s)
    spark        0 hit(s)
    dbutils      0 hit(s)
    mlflow       0 hit(s)

  That hit, in full:
    """The portable core. Knows nothing about Databricks, or any model."""

  It is the docstring SAYING it knows nothing about Databricks.
  grep cannot tell a comment from a dependency. The AST check can.

  sha256[:16] = dd0339d663883037
  Lab 2B recreates this exact file and checks the hash matches.


## 4. Adapter 1 — Chat

*This* is where Databricks appears. The adapter's whole job is to satisfy the
`Chat` protocol using whatever the platform provides.

In [0]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI
import warnings

warnings.filterwarnings("ignore", message=".*serializer warnings.*")

w = WorkspaceClient()
_token = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
_client = OpenAI(api_key=_token, base_url=f"{w.config.host}/serving-endpoints")


class DatabricksChat:
    def __init__(self, model="databricks-claude-sonnet-5"):
        self._model = model

    @property
    def model(self):
        return self._model

    def complete(self, messages, tools):
        return _client.chat.completions.create(
            model=self._model, messages=messages, tools=tools, max_tokens=900
        ).choices[0].message


print(f"  chat adapter ready: {DatabricksChat().model}")

  chat adapter ready: databricks-claude-sonnet-5


## 5. Adapter 2 — Keyword retrieval (TF-IDF)

Deliberately the *least* AI thing that still works: no embeddings, no index, no
network. Pure Python over the `docs` you loaded in cell 1.

Choosing something this different from an embedding model is what makes the
[Lab 2B](#) swap a real test of the interface.

**Note the `audience` filter.** It is enforced *here*, in the adapter — remember
where it lives.

In [0]:
import math
from collections import Counter

_WORD = re.compile(r"[a-z0-9]+")


def _tok(s):
    return _WORD.findall(s.lower())


class KeywordRetriever:
    name = "keyword-tfidf"

    def __init__(self, docs):
        self.docs = docs
        self._tf = [Counter(_tok(f"{d['title']} {d['body']}")) for d in docs]
        df = Counter()
        for t in self._tf:
            df.update(set(t))
        n = len(docs)
        self._idf = {w: math.log((n + 1) / (c + 0.5)) for w, c in df.items()}

    def _score(self, q, tf):
        if not tf:
            return 0.0
        num = sum(tf[w] * self._idf.get(w, 0.0) for w in q)
        norm = math.sqrt(sum(v * v for v in tf.values())) or 1.0
        return num / norm

    def search(self, query, k=3, **filters):
        aud = filters.get("audience")
        q = _tok(query)
        out = [{"doc_id": d["doc_id"], "title": d["title"],
                "text": d["body"], "score": self._score(q, tf)}
               for d, tf in zip(self.docs, self._tf)
               if not aud or d["audience"] == aud]
        out.sort(key=lambda r: r["score"], reverse=True)
        return out[:k]


kw = KeywordRetriever(docs)
for h in kw.search("how long does delivery to the nordics take", audience="customer"):
    print(f"  {h['doc_id']}  score={h['score']:.4f}  {h['title']}")

  DOC-003  score=0.7201  Delivery timescales by region
  DOC-001  score=0.2167  Returns and refunds policy
  DOC-005  score=0.1080  Billing, invoices and payment terms


## 6. Adapter 3 — Order tools, from Unity Catalog

The second source. `search_policy` answers *"what is the rule?"*; this answers
*"what is true about this order?"*

In [0]:
class SparkOrderTools:
    def schemas(self):
        return [{
            "type": "function",
            "function": {
                "name": "get_order",
                "description": "Look up one order and the customer who placed it: status, "
                               "item, revenue, the customer's region and loyalty tier.",
                "parameters": {
                    "type": "object",
                    "properties": {"order_id": {"type": "string",
                                                "description": "e.g. ORD-1044"}},
                    "required": ["order_id"],
                },
            },
        }]

    def call(self, name, args):
        if name != "get_order":
            return {"error": f"unknown tool {name}"}
        oid = str(args.get("order_id", "")).replace("'", "")
        rows = spark.sql(f"""
            SELECT o.order_id, o.status, o.item, o.units, o.revenue, o.order_date,
                   c.name AS customer, c.region, c.tier
            FROM agents_labs.retail.orders o
            JOIN agents_labs.retail.customers c USING (customer_id)
            WHERE o.order_id = '{oid}'
        """).collect()
        return rows[0].asDict() if rows else {"error": f"no such order: {oid}"}


tools = SparkOrderTools()
print(" ", tools.call("get_order", {"order_id": "ORD-1044"}))

  {'order_id': 'ORD-1044', 'status': 'delivered', 'item': 'standing desk', 'units': 5, 'revenue': Decimal('2700.00'), 'order_date': datetime.date(2026, 9, 18), 'customer': 'Helsinki Works', 'region': 'Nordics', 'tier': 'standard'}


## 7. Wire it together

Three objects, none of which `core.py` knows the name of.

In [0]:
import core

chat = DatabricksChat()
print(f"  chat      {chat.model}")
print(f"  retriever {kw.name}")
print(f"  tools     get_order")

  chat      databricks-claude-sonnet-5
  retriever keyword-tfidf
  tools     get_order


## 8. Ask something that needs BOTH sources

> *"My order ORD-1044 hasn't arrived. How long should it take, and what are my
> options?"*

Neither source answers this alone:

- The **order** says where it is going (Nordics) — but not the delivery window.
- The **policy** gives Nordics timescales — but does not know about this order.

**What to watch:** the agent calls `get_order` first, *then* searches policy using
what it learned. That ordering is not in the code — it worked it out.

In [0]:
result = core.run(
    "My order ORD-1044 hasn't arrived. How long should it take, and what are my options?",
    chat=chat, retriever=kw, tools=tools,
)
print("\n" + result["answer"])

  step 1: get_order({"order_id": "ORD-1044"}) -> {"order_id": "ORD-1044", "status": "delivered", "item": "standing desk", "units": 5, "revenue": "2700.00", "or
  step 1: search_policy('delivery time and late delivery options') -> ['DOC-003', 'DOC-001', 'DOC-004']
  step 2: answer

Good news — I checked your order, and ORD-1044 (5 standing desks) is actually already marked as **delivered** in our system.

For reference, here's the delivery timeline that applies to your region:
- Since Helsinki Works is in the **Nordics**, standard delivery takes **5–7 working days**, because shipments are consolidated at our Hamburg hub before onward transport [DOC-003].

Since the system shows it as delivered, here are your options:

1. **Check with your site/reception** — sometimes an order is marked delivered once it reaches a local depot or is signed for by someone else at your location.
2. **If it genuinely hasn't arrived**, this would be treated as a delivery discrepancy rather than a delay — plea

## 9. What did it actually touch?

`core.run` returns a `used` record alongside the answer. This is the beginning of
the audit trail that [Session 3](#) turns into real MLflow traces.

In [0]:
print(f"  retriever : {result['retriever']}")
print(f"  model     : {result['model']}")
print(f"  documents : {result['used']['retrieval']}")
print(f"  tools     : {result['used']['tools']}")

  retriever : keyword-tfidf
  model     : databricks-claude-sonnet-5
  documents : ['DOC-003', 'DOC-001', 'DOC-004']
  tools     : ['get_order']


## 10. What you built

| Piece | Cell | Why it matters |
|---|---|---|
| Three Protocols | 2 | the contract, written before any implementation |
| `core.py` | 3 | the agent logic — standard library only |
| The leak check | 3 | *proof*, not a claim, that it is portable |
| Chat adapter | 4 | where Databricks is allowed to appear |
| Keyword retriever | 5 | the `audience` filter lives **in the adapter** |
| Order tools | 6 | the second source |
| Two-source answer | 8 | the agent ordered the calls itself |
| `used` record | 9 | which documents and tools were actually touched |

### Try this before you move on

1. Ask something needing **only** policy: *"How long do I have to return a chair?"*
   Check `used` — does it still call `get_order`?
2. Ask about **ORD-9999**, which does not exist. Does it invent a delivery date?
3. In cell 5, delete the `if not aud or d["audience"] == aud` filter, then ask
   *"what is the refund approval limit?"*. **The agent-only document now reaches
   the customer.** That single clause is the only thing preventing it — and in
   [Lab 7B](#) a different tool, pointed at the same documents, has no such clause.